# StyleMistral, QLoRA fine-tuning of Mistral-7B-Instruct for outfit recommendations!

Fine-tunes **Mistral-7B-Instruct-v0.3** with **QLoRA** (Dettmers et al., 2023) on the
`neuralwork/fashion-style-instruct` dataset: given a person's body type / personal style and an event,
the model suggests 5 complete outfit combinations.


In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets rouge_score

## 1. Imports, login, settings

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Kaggle "T4 x2": use one GPU (simpler, and the model fits)

import json, math, time
import torch
import transformers, peft, bitsandbytes
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          TrainingArguments, Trainer, DataCollatorForSeq2Seq)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from rouge_score import rouge_scorer
from huggingface_hub import login

# Reads the HF_TOKEN secret on Kaggle or Colab
if os.path.exists("/kaggle"):
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
else:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))

MODEL_ID   = "mistralai/Mistral-7B-Instruct-v0.3"
DATASET_ID = "neuralwork/fashion-style-instruct"
N_TRAIN    = 1500   # training examples (raise if you have a faster GPU)
N_TEST     = 150    # held-out examples for loss / perplexity
N_GEN      = 10     # held-out examples for before/after generation (generation is slow on T4)
MAX_LEN    = 1024   # max tokens per training example
MAX_NEW_TOKENS = 600
SEED       = 42
OUTPUT_DIR = "mistral-7b-style-qlora"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BF16 = torch.cuda.is_bf16_supported()            # False on T4 -> use float16
COMPUTE_DTYPE = torch.bfloat16 if BF16 else torch.float16
print("GPU:", torch.cuda.get_device_name(0), "| compute dtype:", COMPUTE_DTYPE)
print("transformers", transformers.__version__, "| peft", peft.__version__, "| bitsandbytes", bitsandbytes.__version__)

## 2. Load Mistral-7B in 4-bit (NF4 + double quantization)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # QLoRA ingredient 1: 4-bit NormalFloat
    bnb_4bit_use_double_quant=True,     # QLoRA ingredient 2: double quantization
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=COMPUTE_DTYPE,
)
print(f"Model memory: {model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token   # Mistral has no pad token
tokenizer.padding_side = "right"

## 3. Dataset

In [ ]:
data = load_dataset(DATASET_ID)["train"].shuffle(seed=SEED)
print(data)

test_data  = data.select(range(N_TEST))
train_data = data.select(range(N_TEST, N_TEST + N_TRAIN))
print(f"train: {len(train_data)} | test (held-out): {len(test_data)}")

ex = train_data[0]
print("\nINPUT:\n", ex["input"], "\n\nCONTEXT:\n", ex["context"], "\n\nCOMPLETION:\n", ex["completion"][:1000])

## 4. Prompt template
The instruction goes inside Mistral's `[INST] ... [/INST]` chat format (applied by the tokenizer's chat template).

In [ ]:
instructions_string = (
    "You are a personal stylist recommending fashion advice and clothing combinations. "
    "Use the self body and style description below, combined with the event described in the context, "
    "to generate 5 self-contained and complete outfit combinations."
)

def build_prompt(example):
    return f"{instructions_string}\n\n### Input:\n{example['input']}\n\n### Context:\n{example['context']}"

def chat_prompt(example):
    messages = [{"role": "user", "content": build_prompt(example)}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print(chat_prompt(test_data[0]))

## 5. Base model output (before fine-tuning)

In [ ]:
@torch.no_grad()
def generate(example):
    inputs = tokenizer(chat_prompt(example), add_special_tokens=False, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,                 # greedy -> reproducible
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

model.eval()
gen_examples = test_data.select(range(N_GEN))

t0 = time.time()
base_outputs = [generate(e) for e in gen_examples]
print(f"Generated {N_GEN} base outputs in {(time.time() - t0) / 60:.1f} min\n")
print("BASE MODEL OUTPUT (example 0):\n", base_outputs[0])

## 6. Prepare the model for k-bit training

In [ ]:
model.train()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

## 7. LoRA config (all linear layers, as in the QLoRA paper)

In [ ]:
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, config)

trainable_params, all_params = model.get_nb_trainable_parameters()
print(f"trainable params: {trainable_params:,} || all params: {all_params:,} || trainable%: {100 * trainable_params / all_params:.4f}")

## 8. Tokenize
Each example becomes `[INST] prompt [/INST] completion</s>`.
The `labels` for the prompt part are set to `-100`, so the loss is computed **only on the stylist's answer**.

In [ ]:
def tokenize_function(example):
    user = [{"role": "user", "content": build_prompt(example)}]
    full = user + [{"role": "assistant", "content": example["completion"]}]

    prompt_text = tokenizer.apply_chat_template(user, tokenize=False, add_generation_prompt=True)
    full_text   = tokenizer.apply_chat_template(full, tokenize=False)

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids   = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=MAX_LEN)["input_ids"]

    labels = ([-100] * len(prompt_ids) + full_ids[len(prompt_ids):])[:len(full_ids)]
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels,
            "prefix_ok": full_ids[:len(prompt_ids)] == prompt_ids, "length": len(full_ids)}

cols = train_data.column_names
tokenized_train = train_data.map(tokenize_function, remove_columns=cols)
tokenized_test  = test_data.map(tokenize_function, remove_columns=cols)

lengths = tokenized_train["length"]
print(f"tokens per example: mean {sum(lengths) / len(lengths):.0f}, max {max(lengths)}")
print(f"truncated at {MAX_LEN}: {sum(l >= MAX_LEN for l in lengths)} examples")
print(f"prompt/answer split correct: {sum(tokenized_train['prefix_ok'])}/{len(tokenized_train)}")

tokenized_train = tokenized_train.remove_columns(["prefix_ok", "length"])
tokenized_test  = tokenized_test.remove_columns(["prefix_ok", "length"])

# sanity check: the trained-on tokens of example 0 should be exactly the answer
lab = [t for t in tokenized_train[0]["labels"] if t != -100]
print("\nTrained-on text (start):", tokenizer.decode(lab)[:300])

In [ ]:
# pads input_ids with the pad token and labels with -100 (so padding is ignored in the loss)
data_collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)

## 9. Training arguments (QLoRA paper settings: paged optimizer, constant LR, grad-norm clip 0.3)

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,          # effective batch size 16
    num_train_epochs=1,
    lr_scheduler_type="constant_with_warmup",
    warmup_steps=5,
    max_grad_norm=0.3,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=1,
    fp16=not BF16,
    bf16=BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_32bit",              # QLoRA ingredient 3: paged optimizer
    label_names=["labels"],
    report_to="none",
    seed=SEED,
)

## 10. Train
Before training we evaluate once: LoRA's `B` matrices start at zero, so at this point the model is
exactly the base model → this gives the **base model's held-out loss** for a fair before/after comparison.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
)

base_eval = trainer.evaluate()
print(f"Base model held-out loss: {base_eval['eval_loss']:.4f}  (perplexity {math.exp(base_eval['eval_loss']):.2f})")

model.config.use_cache = False
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
train_result = trainer.train()
train_minutes = (time.time() - t0) / 60
peak_mem_gb = torch.cuda.max_memory_allocated() / 1e9
model.config.use_cache = True

ft_eval = trainer.evaluate()
print(f"\nTraining time: {train_minutes:.1f} min | peak GPU memory: {peak_mem_gb:.1f} GB")
print(f"Fine-tuned held-out loss: {ft_eval['eval_loss']:.4f}  (perplexity {math.exp(ft_eval['eval_loss']):.2f})")

## 11. Loss curves

In [ ]:
history = trainer.state.log_history
train_pts = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_pts  = [(0, base_eval["eval_loss"])] + [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(8, 4.5))
plt.plot(*zip(*train_pts), label="train loss")
plt.plot(*zip(*eval_pts), "o-", label="held-out loss")
plt.xlabel("optimizer step"); plt.ylabel("loss"); plt.title("QLoRA fine-tuning: Mistral-7B-Instruct on fashion-style-instruct")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/loss_curve.png", dpi=150)
plt.show()

print("step | train loss")
for s, l in train_pts:
    print(f"{s:4d} | {l:.4f}")

## 12. Fine-tuned model output (same held-out examples)

In [ ]:
model.eval()
t0 = time.time()
ft_outputs = [generate(e) for e in gen_examples]
print(f"Generated {N_GEN} fine-tuned outputs in {(time.time() - t0) / 60:.1f} min\n")

e = gen_examples[0]
print("INPUT:\n", e["input"], "\nCONTEXT:\n", e["context"])
print("\n================ BASE MODEL ================\n", base_outputs[0])
print("\n================ FINE-TUNED ================\n", ft_outputs[0])
print("\n================ REFERENCE ================\n", e["completion"])

## 13. Metrics: ROUGE-L against the reference answers

In [ ]:
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
rouge = lambda preds: sum(scorer.score(e["completion"], p)["rougeL"].fmeasure for e, p in zip(gen_examples, preds)) / len(preds)

base_rouge, ft_rouge = rouge(base_outputs), rouge(ft_outputs)
print(f"ROUGE-L  base: {base_rouge:.4f}  ->  fine-tuned: {ft_rouge:.4f}")

## 14. Save adapter and results

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/adapter")      # LoRA weights only
tokenizer.save_pretrained(f"{OUTPUT_DIR}/adapter")

results = {
    "base_model": MODEL_ID, "dataset": DATASET_ID,
    "n_train": len(train_data), "n_test": len(test_data), "n_generation_eval": N_GEN,
    "trainable_params": trainable_params, "total_params": all_params,
    "trainable_percent": 100 * trainable_params / all_params,
    "train_loss_first": train_pts[0][1], "train_loss_last": train_pts[-1][1],
    "heldout_loss_base": base_eval["eval_loss"], "heldout_loss_finetuned": ft_eval["eval_loss"],
    "perplexity_base": math.exp(base_eval["eval_loss"]), "perplexity_finetuned": math.exp(ft_eval["eval_loss"]),
    "rougeL_base": base_rouge, "rougeL_finetuned": ft_rouge,
    "train_minutes": train_minutes, "peak_gpu_memory_gb": peak_mem_gb,
    "gpu": torch.cuda.get_device_name(0),
    "examples": [{"input": e["input"], "context": e["context"], "base": b, "fine_tuned": f, "reference": e["completion"]}
                 for e, b, f in zip(gen_examples, base_outputs, ft_outputs)],
}
with open(f"{OUTPUT_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2)

# Optional: publish the adapter
# model.push_to_hub("your-username/mistral-7b-style-qlora")

In [ ]:
r = results
drop = 100 * (1 - r["heldout_loss_finetuned"] / r["heldout_loss_base"])
print(f"""| Metric | Base Mistral-7B-Instruct | Fine-tuned (QLoRA) |
|---|---|---|
| Held-out loss ({r['n_test']} examples) | {r['heldout_loss_base']:.3f} | {r['heldout_loss_finetuned']:.3f} |
| Held-out perplexity | {r['perplexity_base']:.2f} | {r['perplexity_finetuned']:.2f} |
| ROUGE-L ({r['n_generation_eval']} examples) | {r['rougeL_base']:.3f} | {r['rougeL_finetuned']:.3f} |

- Training examples: {r['n_train']:,}
- Trainable parameters: {r['trainable_params']:,} of {r['total_params']:,} ({r['trainable_percent']:.2f}%)
- Train loss: {r['train_loss_first']:.3f} -> {r['train_loss_last']:.3f}
- Held-out loss reduction: {drop:.1f}%
- Training time: {r['train_minutes']:.0f} min on {r['gpu']}, peak GPU memory {r['peak_gpu_memory_gb']:.1f} GB
""")